# 01. Hello Crusoe Mill

Crusoe Mill is Crusoe's Tinker-compatible API for running fine-tuning and reinforcement learning experiments: the same client surface and wire protocol as Tinker, so a loop written against one runs against the other. You write the training loop in Python on your own machine. The GPU tasks (forward passes, backward passes, optimizer steps, generation) run on Crusoe.

You keep the logic. Crusoe runs the compute. Nothing here needs a local GPU.

**This notebook**: connect, open a session, generate some text, and read the response
properly. No training yet, this is the client surface you will write training loops
against.

## Setup

Two things to do: install the SDK, and authenticate with an Intelligence API key from the
[Crusoe Console](https://console.crusoecloud.com/).

`crusoe-mill` is in limited preview and not on a public package index yet, so it ships next
to this notebook as a wheel in `sdk/`. `%pip` installs it into the interpreter running **this
notebook**, which is what keeps the import below from landing in a different environment than
the kernel.

The two lines above it are only there because `uv`-created virtual environments ship without
`pip`, and `%pip` needs it. `ensurepip` is in the standard library and carries its own copy,
so nothing extra is downloaded.

In [17]:
try:
    import pip
except ImportError:
    import ensurepip
    ensurepip.bootstrap()

%pip install --quiet --upgrade -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Now the credentials. The cell below reads `CRUSOE_API_KEY` from your environment and prompts
for it if it is not set, so the key never gets saved into this notebook file.

`CRUSOE_BASE_URL` has to be set explicitly: the SDK's built-in default endpoint is not the one
serving this preview.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("CRUSOE_API_KEY"):
    os.environ["CRUSOE_API_KEY"] = getpass("Crusoe API key: ")

os.environ.setdefault("CRUSOE_BASE_URL", "https://api.intelligence.crusoecloud.com")

print("endpoint:", os.environ["CRUSOE_BASE_URL"])

## The client hierarchy

Three objects, and you will use all three in every notebook that follows:

| object | what it does |
|---|---|
| `ServiceClient` | your connection. Creates the other two. |
| `SamplingClient` | generates text from a model. |
| `TrainingClient` | runs `forward_backward` and `optim_step`. |

One point to know up front: a `ServiceClient` opens a **session**, and the
session declares which models it needs when it is created. That is how the service reserves
GPUs for you.

Declare a row for each model you will use: `training_client_base_models` for one you will
train, `sampling_client_base_models` for one you will sample from. Both lists are optional, so
this notebook declares a sampler and no trainer.

Each row also takes **`num_replicas`**, which sets how many nodes are allocated to that base
model. Training and sampling throughput scale roughly in proportion to it, so you can choose
both the overall throughput of your run and the ratio between training and sampling. Left
unset, each row gets one replica. An RL loop that spends most of its time generating rollouts,
for example, might double its samplers:

```python
ServiceConfig(
    training_client_base_models=[TrainingBaseModelConfig(base_model=MODEL, num_replicas=1)],
    sampling_client_base_models=[SamplingBaseModelConfig(base_model=MODEL, num_replicas=2)],
)
```

In [3]:
from crusoe_mill import SamplingBaseModelConfig, ServiceClient, ServiceConfig

MODEL = "Qwen/Qwen3.8-27B"

client = ServiceClient(
    service_config=ServiceConfig(
        sampling_client_base_models=[SamplingBaseModelConfig(base_model=MODEL)],
    ),
)
print("session declared for", MODEL)

[2026-09-28 21:37:26 crusoe_mill._progress INFO] tinker session msess_512b9d75b36240adbd7c65a762d5815a created; provisioning Qwen/Qwen3.8-27B
[2026-09-28 21:37:26 crusoe_mill.service_client INFO] ServiceClient initialized for session msess_512b9d75b36240adbd7c65a762d5815a
[2026-09-28 21:37:27 crusoe_mill._progress INFO] provisioning — 3s elapsed (session msess_512b9d75b36240adbd7c65a762d5815a; ^C releases it)
[2026-09-28 21:38:27 crusoe_mill._progress INFO] provisioning — 63s elapsed (session msess_512b9d75b36240adbd7c65a762d5815a; ^C releases it)
[2026-09-28 21:39:30 crusoe_mill._progress INFO] provisioning — 126s elapsed (session msess_512b9d75b36240adbd7c65a762d5815a; ^C releases it)
[2026-09-28 21:40:33 crusoe_mill._progress INFO] provisioning — 189s elapsed (session msess_512b9d75b36240adbd7c65a762d5815a; ^C releases it)


session declared for Qwen/Qwen3.8-27B


## Waiting for the session

**The cell above is the wait.** Constructing the `ServiceClient` with a `ServiceConfig`
declares the session, starts provisioning, and does not return until the GPUs are ready, so on
a cold service it blocks. It prints progress while it waits,
and interrupting it releases the session. If the fleet is busy the scheduler may queue you
before any of that starts.

Pass `wait_for_ready=False` to `ServiceClient` if you would rather it return immediately and
take the wait on your first GPU-backed call instead.

In [4]:
from tinker import types

sampling_client = client.create_sampling_client(base_model=MODEL)
tokenizer = sampling_client.get_tokenizer()
print("ready")

ready


## Generating text

Mill works in **tokens**, not strings. The service never sees your text: you encode it,
send token ids, and decode what comes back. That is a deliberate part of the design and
what lets you control masking and loss boundaries exactly when you start training.

So sampling is three steps:

1. encode your prompt and wrap it in a `ModelInput`
2. `sample()` it with some `SamplingParams`
3. decode the tokens that come back

The prompt here is a bare sentence rather than a question, to show the plainest
possible token-in / token-out path. `Qwen3.8-27B` is instruction-tuned, so expect it to answer
in a conversational manner rather than simply continuing the sentence - the chat template
further down is the right way to talk to it.

In [13]:
prompt_text = "The three largest cities in the world by population are"
prompt = types.ModelInput.from_ints(tokenizer.encode(prompt_text))

result = sampling_client.sample(
    prompt=prompt,
    sampling_params=types.SamplingParams(max_tokens=7, temperature=0),
    num_samples=1,
).result()

print(prompt_text + tokenizer.decode(result.sequences[0].tokens))

The three largest cities in the world by population are Tokyo, Delhi, and Shanghai.


`sample()` returns a **future**, and `.result()` is where you wait. That matters more than it
looks: you can fire off many `sample()` calls and only then collect them, which is how the RL
loop in the next notebook generates a whole batch concurrently.

### Reading the response

Each sampled sequence carries more than text. `logprobs` is needed by the RL loop because it records what the sampling policy believed at the
moment it generated each token.

In [14]:
sequence = result.sequences[0]

print("stop reason :", sequence.stop_reason)
print("tokens      :", len(sequence.tokens))
print("token ids   :", sequence.tokens[:10], "...")
print("logprobs    :", [round(lp, 3) for lp in sequence.logprobs[:10]], "...")

stop reason : length
tokens      : 7
token ids   : [25358, 11, 21318, 11, 321, 35817, 13] ...
logprobs    : [-0.52, -0.353, -1.012, -0.058, -0.011, -0.067, -0.279] ...


## Chat models take a chat template

The prompt above was raw text, which is fine for a base model. An instruct model expects its
turns wrapped in the template it was trained with. `apply_chat_template` does that, and getting it wrong is a silent failure: the
model still answers, just worse, and nothing tells you why.

In [15]:
messages = [
    {"role": "system", "content": "You are terse. Answer in one sentence."},
    {"role": "user", "content": "Why is the sky blue?"},
]

prompt_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=False,
    enable_thinking=False,
)
print(repr(tokenizer.decode(prompt_ids)), "\n")

chat_prompt = types.ModelInput.from_ints(prompt_ids)

result = sampling_client.sample(
    prompt=chat_prompt,
    sampling_params=types.SamplingParams(max_tokens=100, temperature=0.7),
    num_samples=1,
).result()

print(tokenizer.decode(result.sequences[0].tokens, skip_special_tokens=True))

'<|im_start|>system\nYou are terse. Answer in one sentence.<|im_end|>\n<|im_start|>user\nWhy is the sky blue?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n' 

The sky appears blue because shorter wavelengths of sunlight, such as blue, are scattered more effectively by the atmosphere's molecules than longer wavelengths like red.


## Close the session

A session holds GPUs while it is alive, so close it when you are done. Until it goes, its
model stays resident and the next run you start may sit waiting for capacity your own last
notebook is still holding. That matters more here than on a small model: the session above
holds a 27B sampler.

There are two ways it ends. `close()` releases it immediately and is idempotent. If your
script simply exits, the cluster spins down on its own about **15 minutes** later.

While a session is still alive you can re-attach to it
from a new script by passing its id as `existing_session_id` in `ServiceConfig`, which skips
provisioning entirely. Useful when a notebook dies mid-run and you do not want to wait out a
cold start again.

In [16]:
client.close()
print("session closed")

session closed


## What you have now

Everything in this notebook is the client surface a training loop is built on: a session that
declares its models, a sampler you can call concurrently, token ids in and token ids out, and
per-token logprobs alongside every completion.

The pieces a training loop adds on top are a `TrainingClient` from the same `ServiceClient`,
`forward_backward` and `optim_step` to push gradients, and a call to publish the updated
weights so the next round of sampling uses them. There are two of those:
`save_weights_and_get_sampling_client` hands you a ready sampler in one step, while
`save_weights_for_sampler` returns the checkpoint path instead, which you keep if you
want to sample from that exact version again later.